# MD Python V2 API smoke test

Manual end-to-end smoke test for the V2 client surface after the `feature/extra-endpoints` merge. Run top-to-bottom. Every section prints `[OK]`, `[FAIL]` or `[SKIP]` so a human can eyeball what passed.

**Prereqs:**
- `.env` with `MD_AUTH_TOKEN` (and optionally `MD_API_BASE_URL`).
- The repo `.venv` as the notebook kernel.

Each section wraps its calls in `try/except` so a failure in one section does not stop the rest of the notebook.

## Section 0: Setup

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception as _exc:
    print(f"[SKIP] dotenv not available: {_exc}")

from md_python.client_v2 import MDClientV2
from md_python.models import Upload, Dataset, SampleMetadata, ExperimentDesign

# Status tracking for the final summary
RESULTS: dict[str, tuple[str, str]] = {}

def section(title: str) -> None:
    bar = "=" * 72
    print(f"\n{bar}\n{title}\n{bar}")

def status(ok, section_name: str, message: str = "") -> None:
    if ok is True:
        tag = "[OK]"
    elif ok is False:
        tag = "[FAIL]"
    else:
        tag = "[SKIP]"
    RESULTS[section_name] = (tag, message)
    print(f"{tag} {section_name}: {message}")

def mask(token):
    if not token:
        return "<missing>"
    if len(token) <= 8:
        return "*" * len(token)
    return f"{token[:4]}...{token[-4:]}"

client = MDClientV2()
print(f"base_url: {client.base_url}")
print(f"token:    {mask(os.getenv('MD_AUTH_TOKEN'))}")

## Section 1: Health check

In [ ]:
section("Section 1: Health check")
try:
    health = client.health.check()
    print("Response:", health)
    status(True, "1_health", str(health))
except Exception as exc:
    status(False, "1_health", f"{type(exc).__name__}: {exc}")

## Section 3: Query uploads with filters

In [ ]:
section("Section 3: Query uploads")

# 3a: search by name
try:
    upload_id = "6db6e841-a1ef-4d74-b920-c6be7b581fb1"
    results = client.uploads.query(search="CKD")
    data = results.get("data", [])
    print(f"search='CKD' -> {len(data)} hits (first page)")
    if upload_id is not None and any(str(u.get('id')) == str(upload_id) for u in data):
        status(True, "3a_query_search", f"upload {upload_id} present")
    elif upload_id is None:
        status(None, "3a_query_search", "no upload_id from Section 2")
    else:
        status(False, "3a_query_search", f"upload {upload_id} not in search results")
except Exception as exc:
    status(False, "3a_query_search", f"{type(exc).__name__}: {exc}")

# 3b: filter by status=completed, page 1
try:
    completed_page = client.uploads.query(status=["completed"], page=1)
    data = completed_page.get("data", [])
    print(f"status=['completed'] page=1 -> {len(data)} hits")
    for u in data[:10]:
        print(f"  - {u.get('name')} ({u.get('status')})")
    status(True, "3b_query_status", f"{len(data)} completed uploads on page 1")
except Exception as exc:
    status(False, "3b_query_status", f"{type(exc).__name__}: {exc}")

# 3c: server-side sample_metadata column filter (new)
try:
    meta_results = client.uploads.query(
        sample_metadata=[{"column": "condition", "value": "Stage 1"}],
    )
    data = meta_results.get("data", [])
    print(f"sample_metadata condition=treated -> {len(data)} hits")
    status(True, "3c_query_sample_metadata", f"{len(data)} matches")
except Exception as exc:
    status(False, "3c_query_sample_metadata", f"{type(exc).__name__}: {exc}")

## Section 4: Round-trip upload sample metadata

In [ ]:
SAMPLE_NAMES = ["LFQ intensity CKD_1_1","LFQ intensity CKD_1_2"]
section("Section 4: get_sample_metadata round-trip")
try:
    if upload_id is None:
        status(None, "4_get_sample_metadata", "no upload_id from Section 2")
    else:
        sm = client.uploads.get_sample_metadata(upload_id)
        if sm is None:
            status(False, "4_get_sample_metadata", "server returned None")
        else:
            print("Returned sample_metadata.data:")
            for row in sm.data:
                print(f"  {row}")
            header = sm.data[0]
            try:
                sn_idx = header.index("sample_name")
            except ValueError:
                sn_idx = 0
            returned_names = {r[sn_idx] for r in sm.data[1:] if len(r) > sn_idx}
            if returned_names == set(SAMPLE_NAMES):
                status(True, "4_get_sample_metadata", f"sample_names match: {returned_names}")
            else:
                status(False, "4_get_sample_metadata",
                       f"mismatch: got {returned_names}, expected {set(SAMPLE_NAMES)}")
except Exception as exc:
    status(False, "4_get_sample_metadata", f"{type(exc).__name__}: {exc}")

## Section 5: `get_by_name` replacement via query + local filter

`uploads.get_by_name` was removed in the `feature/extra-endpoints` merge. This section demonstrates the replacement pattern: `uploads.query(search=...)` followed by a local exact-name filter.

In [ ]:
section("Section 5: find_upload_by_name helper")

UPLOAD_NAME = "CKD Demo 2025"

def find_upload_by_name(name: str):
    """Replacement for the removed uploads.get_by_name.

    Uses the server-side search filter and then matches exact name locally.
    Walks pages until the search returns an empty page or a short page.
    """
    page = 1
    while True:
        resp = client.uploads.query(search=name, page=page)
        data = resp.get("data", [])
        if not data:
            return None
        for u in data:
            if u.get("name") == name:
                return u
        if len(data) < 10:
            return None
        page += 1
        if page > 20:
            return None

try:
    found = find_upload_by_name(UPLOAD_NAME)
    if found is None:
        status(False, "5_find_by_name", f"helper could not locate {UPLOAD_NAME}")
    else:
        print(f"Found: id={found.get('id')} name={found.get('name')} status={found.get('status')}")
        status(True, "5_find_by_name", f"id={found.get('id')}")
except Exception as exc:
    status(False, "5_find_by_name", f"{type(exc).__name__}: {exc}")

## Section 6: Dataset query / show / download

In [ ]:
section("Section 6: dataset query / get_by_id / download_table_url")

intensity_id = None

# 6a: query datasets for the upload
try:
    if upload_id is None:
        status(None, "6a_datasets_query", "no upload_id from Section 2")
    else:
        dq = client.datasets.query(upload_id=upload_id)
        ds_list = dq.get("data", [])
        print(f"datasets.query(upload_id=...) -> {len(ds_list)} datasets")
        for d in ds_list:
            print(f"  - id={d.get('id')} name={d.get('name')} type={d.get('type')} state={d.get('state')}")
        intensity = [d for d in ds_list if d.get("type") == "INTENSITY"]
        if not intensity:
            status(False, "6a_datasets_query", "no INTENSITY dataset found")
        else:
            intensity_id = str(intensity[0]["id"])
            status(True, "6a_datasets_query", f"intensity_id={intensity_id}")
except Exception as exc:
    status(False, "6a_datasets_query", f"{type(exc).__name__}: {exc}")

# 6b: get_by_id
try:
    if intensity_id is None:
        status(None, "6b_datasets_get_by_id", "no intensity_id")
    else:
        ds = client.datasets.get_by_id(intensity_id)
        print(ds)
        status(True, "6b_datasets_get_by_id", f"state={getattr(ds, 'state', None)}")
except Exception as exc:
    status(False, "6b_datasets_get_by_id", f"{type(exc).__name__}: {exc}")

# 6c: download_table_url (csv)
csv_url = None
try:
    if intensity_id is None:
        status(None, "6c_download_csv_url", "no intensity_id")
    else:
        csv_url = client.datasets.download_table_url(
            intensity_id, "Protein_Metadata", format="csv"
        )
        print(f"csv URL: {csv_url[:120]}...")
        status(True, "6c_download_csv_url", "presigned URL returned")
except Exception as exc:
    status(False, "6c_download_csv_url", f"{type(exc).__name__}: {exc}")

# 6d: download_table_url (parquet)
try:
    if intensity_id is None:
        status(None, "6d_download_parquet_url", "no intensity_id")
    else:
        pq_url = client.datasets.download_table_url(
            intensity_id, "Protein_Metadata", format="parquet"
        )
        print(f"parquet URL: {pq_url[:120]}...")
        status(True, "6d_download_parquet_url", "presigned URL returned")
except Exception as exc:
    status(False, "6d_download_parquet_url", f"{type(exc).__name__}: {exc}")

# 6e: optional - fetch first 200 bytes of the csv URL
try:
    if csv_url is None:
        status(None, "6e_fetch_csv_preview", "no csv_url")
    else:
        import requests
        r = requests.get(csv_url, stream=True, timeout=30)
        snippet = next(r.iter_content(chunk_size=200), b"")
        print(f"HTTP {r.status_code}")
        print(f"First 200 bytes: {snippet[:200]!r}")
        status(True, "6e_fetch_csv_preview", f"HTTP {r.status_code}")
except Exception as exc:
    status(False, "6e_fetch_csv_preview", f"{type(exc).__name__}: {exc}")

## Section 7: Entities query (replaces `entities.search`)

In [ ]:
intensity_id = "fb3e4141-926e-4b97-bf20-e29d6d818570"

In [ ]:
section("Section 7: entities.query")

try:
    if intensity_id is None:
        status(None, "7_entities_query", "no intensity_id from Section 6")
    else:
        # entities.query requires keyword >= 2 chars. Our synthetic protein IDs
        # all start with 'P00' so this should match P00001..P00005.
        resp = client.entities.query(keyword="P00", dataset_ids=[intensity_id])
        print(json.dumps(resp, indent=2)[:1000])
        results = resp.get("results", resp) if isinstance(resp, dict) else resp
        count = len(results) if hasattr(results, "__len__") else "?"
        status(True, "7_entities_query", f"keyword=P00 returned {count} results")
except Exception as exc:
    status(False, "7_entities_query", f"{type(exc).__name__}: {exc}")

## Section 8: Dataset sample-name validation (new server-side check)

The V2 API validates `experiment_design.sample_name` in `job_run_params` against
the upload's stored sample metadata. Any name not in the upload is rejected with
`"Invalid sample names: X, Y. These samples do not exist in the experiment's sample metadata."`

This validation fires for **pairwise_comparison** and **anova** (which send
`experiment_design: {sample_name: [...], condition: [...]}`) but is **skipped**
for **normalisation_imputation** (which does not include `experiment_design` in
its `job_run_params`).

Sub-sections:
- **8a/8b**: Pairwise comparison — bad sample names (should fail) + good (should succeed)
- **8c/8d**: Normalisation + imputation — bad sample names passed as `experiment_design`
  (should be ignored since NI uses a different param shape) + good (should succeed)


In [ ]:
section("Section 8: sample-name validation on datasets.create")

# Section 8 is self-contained: resolve intensity_id and sample_names directly
# from the upload so it can be re-run without Sections 4/6.
upload_id = "6db6e841-a1ef-4d74-b920-c6be7b581fb1"
good_dataset_id = None
TS = datetime.utcnow().strftime("%Y%m%dT%H%M%S")

# Resolve the INTENSITY dataset for this upload
try:
    dq = client.datasets.query(upload_id=upload_id)
    ds_list = dq.get("data", [])
    intensity = [d for d in ds_list if d.get("type") == "INTENSITY"]
    intensity_id = str(intensity[0]["id"]) if intensity else None
    print(f"resolved intensity_id={intensity_id}")
except Exception as exc:
    intensity_id = None
    print(f"failed to resolve intensity_id: {type(exc).__name__}: {exc}")

# Resolve real sample_names from the upload's sample_metadata
try:
    sm = client.uploads.get_sample_metadata(upload_id)
    header = sm.data[0]
    try:
        sn_idx = header.index("sample_name")
    except ValueError:
        sn_idx = 0
    SAMPLE_NAMES = [r[sn_idx] for r in sm.data[1:] if len(r) > sn_idx]
    print(f"resolved {len(SAMPLE_NAMES)} sample_names: {SAMPLE_NAMES}")
except Exception as exc:
    SAMPLE_NAMES = []
    print(f"failed to resolve sample_names: {type(exc).__name__}: {exc}")

# 8a: bogus sample_names -> must fail
try:
    if intensity_id is None:
        status(None, "8a_bad_sample_names", "no intensity_id resolved")
    else:
        bogus = Dataset(
            input_dataset_ids=[intensity_id],
            name=f"smoke-bad-{TS}",
            job_slug="normalisation_imputation",
            job_run_params={
                "entity_type": "protein",
                "dataset_name": f"smoke-bad-{TS}",
                "normalisation_methods": "skip",
                "imputation_methods": "mnar",
                "sample_names": ["NOT_IN_UPLOAD"],
            },
        )
        raised = False
        try:
            client.datasets.create(bogus)
        except Exception as exc:
            raised = True
            msg = str(exc).lower()
            print(f"create raised as expected: {exc}")
            if "sample" in msg and ("not found" in msg or "invalid" in msg or "does not" in msg or "unknown" in msg):
                status(True, "8a_bad_sample_names", "raised with sample-not-found style error")
            else:
                status(True, "8a_bad_sample_names", f"raised: {exc}")
        if not raised:
            status(False, "8a_bad_sample_names", "create did NOT raise for bogus sample_names")
except Exception as exc:
    status(False, "8a_bad_sample_names", f"{type(exc).__name__}: {exc}")

# 8b: correct sample_names -> should succeed; cancel so it does not actually run
try:
    if intensity_id is None:
        status(None, "8b_good_sample_names", "no intensity_id resolved")
    elif not SAMPLE_NAMES:
        status(None, "8b_good_sample_names", "no sample_names resolved from upload")
    else:
        good = Dataset(
            input_dataset_ids=[intensity_id],
            name=f"smoke-good-{TS}",
            job_slug="normalisation_imputation",
            job_run_params={
                "entity_type": "protein",
                "dataset_name": f"smoke-good-{TS}",
                "normalisation_methods": "skip",
                "imputation_methods": "mnar",
                "sample_names": SAMPLE_NAMES,
            },
        )
        good_dataset_id = client.datasets.create(good)
        print(f"Created dataset with valid sample_names: {good_dataset_id}")
        try:
            client.datasets.cancel(good_dataset_id)
            print(f"Cancelled dataset {good_dataset_id}")
        except Exception as cexc:
            print(f"(non-fatal) cancel failed: {cexc}")
        status(True, "8b_good_sample_names", f"dataset_id={good_dataset_id}")
except Exception as exc:
    status(False, "8b_good_sample_names", f"{type(exc).__name__}: {exc}")


In [ ]:
section("Section 8a/8b: Pairwise sample-name validation")

# Re-resolve intensity_id and sample_names for this upload
upload_id = "6db6e841-a1ef-4d74-b920-c6be7b581fb1"
TS = datetime.utcnow().strftime("%Y%m%dT%H%M%S")

try:
    dq = client.datasets.query(upload_id=upload_id)
    ds_list = dq.get("data", [])
    intensity = [d for d in ds_list if d.get("type") == "INTENSITY"]
    intensity_id = str(intensity[0]["id"]) if intensity else None
    print(f"intensity_id={intensity_id}")
except Exception as exc:
    intensity_id = None
    print(f"failed to resolve intensity_id: {exc}")

try:
    sm = client.uploads.get_sample_metadata(upload_id)
    header = sm.data[0]
    sn_idx = header.index("sample_name")
    SAMPLE_NAMES = [r[sn_idx] for r in sm.data[1:] if len(r) > sn_idx]
    CONDITIONS = None
    try:
        cond_idx = header.index("condition")
        CONDITIONS = [r[cond_idx] for r in sm.data[1:] if len(r) > cond_idx]
    except ValueError:
        pass
    print(f"sample_names={SAMPLE_NAMES}, conditions={CONDITIONS}")
except Exception as exc:
    SAMPLE_NAMES = []
    CONDITIONS = None
    print(f"failed to resolve metadata: {exc}")

# 8a: pairwise with BOGUS sample names in experiment_design -> should FAIL
try:
    if intensity_id is None:
        status(None, "8a_pairwise_bad_samples", "no intensity_id")
    else:
        bogus_pc = Dataset(
            input_dataset_ids=[intensity_id],
            name=f"smoke-pc-bad-{TS}",
            job_slug="pairwise_comparison",
            job_run_params={
                "experiment_design": {
                    "sample_name": ["BOGUS_A", "BOGUS_B", "BOGUS_C"],
                    "condition": ["ctrl", "ctrl", "treat"],
                },
                "condition_column": "condition",
                "condition_comparisons": {
                    "condition_comparison_pairs": [["treat", "ctrl"]]
                },
                "filter_values_criteria": {
                    "method": "percentage",
                    "filter_threshold_percentage": 0.5,
                },
                "limma_trend": True,
                "robust_empirical_bayes": True,
                "fit_separate_models": True,
            },
        )
        raised = False
        try:
            client.datasets.create(bogus_pc)
        except Exception as exc:
            raised = True
            msg = str(exc).lower()
            print(f"Pairwise with bogus samples raised: {exc}")
            if "invalid sample" in msg or "do not exist" in msg:
                status(True, "8a_pairwise_bad_samples", "correctly rejected invalid sample names")
            else:
                status(True, "8a_pairwise_bad_samples", f"raised (unexpected wording): {exc}")
        if not raised:
            status(False, "8a_pairwise_bad_samples", "create did NOT raise for bogus sample_names")
except Exception as exc:
    status(False, "8a_pairwise_bad_samples", f"{type(exc).__name__}: {exc}")

# 8b: pairwise with CORRECT sample names -> should succeed; cancel immediately
try:
    if intensity_id is None or not SAMPLE_NAMES or not CONDITIONS:
        status(None, "8b_pairwise_good_samples", "no intensity_id or sample data")
    else:
        unique_conds = list(dict.fromkeys(CONDITIONS))  # preserve order, dedupe
        if len(unique_conds) < 2:
            status(None, "8b_pairwise_good_samples", f"need >=2 conditions, got {unique_conds}")
        else:
            good_pc = Dataset(
                input_dataset_ids=[intensity_id],
                name=f"smoke-pc-good-{TS}",
                job_slug="pairwise_comparison",
                job_run_params={
                    "experiment_design": {
                        "sample_name": SAMPLE_NAMES,
                        "condition": CONDITIONS,
                    },
                    "condition_column": "condition",
                    "condition_comparisons": {
                        "condition_comparison_pairs": [[unique_conds[0], unique_conds[1]]]
                    },
                    "filter_values_criteria": {
                        "method": "percentage",
                        "filter_threshold_percentage": 0.5,
                    },
                    "limma_trend": True,
                    "robust_empirical_bayes": True,
                    "fit_separate_models": True,
                },
            )
            pc_id = client.datasets.create(good_pc)
            print(f"Pairwise created OK: {pc_id}")
            try:
                client.datasets.cancel(pc_id)
                print(f"Cancelled {pc_id}")
            except Exception as cexc:
                print(f"(non-fatal) cancel failed: {cexc}")
            status(True, "8b_pairwise_good_samples", f"dataset_id={pc_id}")
except Exception as exc:
    status(False, "8b_pairwise_good_samples", f"{type(exc).__name__}: {exc}")


In [ ]:
section("Section 8c/8d: NI sample-name validation (expected: validation SKIPPED)")

# NI does not send experiment_design in job_run_params, so the server-side
# ValidateSampleNames.validate returns early (experiment_design is not a Hash).
# Bogus sample names passed via a separate key should be silently ignored.

# 8c: NI with bogus experiment_design containing bad sample_name
# The NI schema doesn't use experiment_design, so the server should either
# ignore it or — if it IS a Hash — validate and reject. Let's test both shapes.
try:
    if intensity_id is None:
        status(None, "8c_ni_bad_samples", "no intensity_id")
    else:
        # Shape A: pass experiment_design as a dict with bad sample_name
        # This MAY trigger validation (experiment_design is a Hash)
        bogus_ni = Dataset(
            input_dataset_ids=[intensity_id],
            name=f"smoke-ni-bad-{TS}",
            job_slug="normalisation_imputation",
            job_run_params={
                "entity_type": "protein",
                "normalisation_methods_proteomics": "skip",
                "imputation_methods": "mnar",
                "experiment_design": {
                    "sample_name": ["BOGUS_NOT_REAL"],
                },
            },
        )
        raised = False
        try:
            bad_ni_id = client.datasets.create(bogus_ni)
        except Exception as exc:
            raised = True
            msg = str(exc).lower()
            print(f"NI with bogus experiment_design.sample_name raised: {exc}")
            if "invalid sample" in msg or "do not exist" in msg:
                status(True, "8c_ni_bad_samples", "server validated experiment_design even for NI")
            else:
                status(True, "8c_ni_bad_samples", f"raised (unexpected): {exc}")
        if not raised:
            print(f"NI accepted bogus experiment_design — validation skipped as expected: {bad_ni_id}")
            status(True, "8c_ni_bad_samples", "validation skipped (expected for NI) — cancel & clean up")
            try:
                client.datasets.cancel(bad_ni_id)
            except Exception:
                pass
except Exception as exc:
    status(False, "8c_ni_bad_samples", f"{type(exc).__name__}: {exc}")

# 8d: NI with correct params (no experiment_design) -> should always succeed
try:
    if intensity_id is None:
        status(None, "8d_ni_good", "no intensity_id")
    else:
        good_ni = Dataset(
            input_dataset_ids=[intensity_id],
            name=f"smoke-ni-good-{TS}",
            job_slug="normalisation_imputation",
            job_run_params={
                "entity_type": "protein",
                "normalisation_methods_proteomics": "skip",
                "imputation_methods": "mnar",
                "std_position": 1.8,
                "std_width": 0.3,
            },
        )
        ni_id = client.datasets.create(good_ni)
        print(f"NI created OK: {ni_id}")
        try:
            client.datasets.cancel(ni_id)
            print(f"Cancelled {ni_id}")
        except Exception as cexc:
            print(f"(non-fatal) cancel failed: {cexc}")
        status(True, "8d_ni_good", f"dataset_id={ni_id}")
except Exception as exc:
    status(False, "8d_ni_good", f"{type(exc).__name__}: {exc}")


## Section 8.5: NI with flat `normalisation_methods` key (not `_proteomics` / `_gene`)

Test whether the API accepts the simpler `normalisation_methods: "median"` shape
instead of `normalisation_methods_proteomics: "median"`.

Section 8d proved that `normalisation_methods_proteomics` works. This section
checks the shorter key name so we know which one the builder should emit.

In [ ]:
section("Section 8.5: NI with flat normalisation_methods key")

upload_id = "6db6e841-a1ef-4d74-b920-c6be7b581fb1"
intensity_id = "1e97e2ae-1ad6-4d3b-a772-f2d496d152fa"
TS = datetime.utcnow().strftime("%Y%m%dT%H%M%S")

# 8.5a: normalisation_methods (flat string, no _proteomics suffix) + MNAR defaults
try:
    flat_ni = Dataset(
        input_dataset_ids=[intensity_id],
        name=f"smoke-ni-flat-{TS}",
        job_slug="normalisation_imputation",
        job_run_params={
            "entity_type": "protein",
            "normalisation_methods": "What ever you want",
            "imputation_methods": "mnar",
            "std_position": 1.8,
            "std_width": 0.3,
        },
    )
    ni_id = client.datasets.create(flat_ni)
    print(f"NI with flat normalisation_methods created OK: {ni_id}")
    # try:
    #     client.datasets.cancel(ni_id)
    #     print(f"Cancelled {ni_id}")
    # except Exception as cexc:
    #     print(f"(non-fatal) cancel failed: {cexc}")
    status(True, "8.5a_ni_flat_key", f"dataset_id={ni_id}")
except Exception as exc:
    status(False, "8.5a_ni_flat_key", f"{type(exc).__name__}: {exc}")

# # 8.5b: compare with normalisation_methods_proteomics (the current builder key) + MNAR defaults
# try:
#     suffixed_ni = Dataset(
#         input_dataset_ids=[intensity_id],
#         name=f"smoke-ni-proteomics-{TS}",
#         job_slug="normalisation_imputation",
#         job_run_params={
#             "entity_type": "protein",
#             "normalisation_methods_proteomics": "median",
#             "imputation_methods": "mnar",
#             "std_position": 1.8,
#             "std_width": 0.3,
#         },
#     )
#     ni_id2 = client.datasets.create(suffixed_ni)
#     print(f"NI with normalisation_methods_proteomics created OK: {ni_id2}")
#     # try:
#     #     client.datasets.cancel(ni_id2)
#     #     print(f"Cancelled {ni_id2}")
#     # except Exception as cexc:
#     #     print(f"(non-fatal) cancel failed: {cexc}")
#     status(True, "8.5b_ni_proteomics_key", f"dataset_id={ni_id2}")
# except Exception as exc:
#     status(False, "8.5b_ni_proteomics_key", f"{type(exc).__name__}: {exc}")

## Section 9: Delete upload (retry after deleting datasets)

In [ ]:
section("Section 9: delete upload (expect 409, then succeed)")
upload_id = "bb524bcf-283a-4571-814b-9686a2001c6d"
# 9a: first delete attempt -> should 409 due to associated datasets
try:
    if upload_id is None:
        status(None, "9a_delete_conflict", "no upload_id from Section 2")
    else:
        try:
            client.uploads.delete(upload_id)
            status(False, "9a_delete_conflict", "delete unexpectedly succeeded (no 409)")
        except Exception as exc:
            msg = str(exc)
            print(f"delete raised as expected: {msg}")
            if "409" in msg:
                status(True, "9a_delete_conflict", "409 raised")
            else:
                status(True, "9a_delete_conflict", f"raised (no explicit 409 in text): {msg}")
except Exception as exc:
    status(False, "9a_delete_conflict", f"{type(exc).__name__}: {exc}")

# 9b: delete each dataset belonging to the upload, then delete the upload
try:
    if upload_id is None:
        status(None, "9b_delete_datasets_then_upload", "no upload_id from Section 2")
    else:
        dq = client.datasets.query(upload_id=upload_id)
        ds_list = dq.get("data", [])
        for d in ds_list:
            did = str(d.get("id"))
            try:
                client.datasets.delete(did)
                print(f"Deleted dataset {did}")
            except Exception as dexc:
                print(f"(non-fatal) failed to delete {did}: {dexc}")

        client.uploads.delete(upload_id)
        print(f"Deleted upload {upload_id}")
        status(True, "9b_delete_datasets_then_upload", f"upload {upload_id} deleted")
except Exception as exc:
    status(False, "9b_delete_datasets_then_upload", f"{type(exc).__name__}: {exc}")

# 9c: verify upload is gone
try:
    if upload_id is None:
        status(None, "9c_upload_gone", "no upload_id")
    else:
        try:
            leftover = client.uploads.get_by_id(upload_id)
            if leftover is None:
                status(True, "9c_upload_gone", "get_by_id returned None")
            else:
                status(False, "9c_upload_gone", f"upload still present: status={leftover.status}")
        except Exception as exc:
            print(f"get_by_id raised as expected: {exc}")
            status(True, "9c_upload_gone", f"get_by_id raised: {type(exc).__name__}")
except Exception as exc:
    status(False, "9c_upload_gone", f"{type(exc).__name__}: {exc}")

## Section 10: Summary

In [ ]:
section("Section 10: Summary")

width = max((len(k) for k in RESULTS), default=10)
for key in sorted(RESULTS):
    tag, msg = RESULTS[key]
    print(f"{tag:<6} {key:<{width}} :: {msg}")

totals = {"[OK]": 0, "[FAIL]": 0, "[SKIP]": 0}
for tag, _ in RESULTS.values():
    totals[tag] = totals.get(tag, 0) + 1
print()
print(
    f"Totals: OK={totals['[OK]']}  FAIL={totals['[FAIL]']}  SKIP={totals['[SKIP]']}  "
    f"(of {len(RESULTS)} checks)"
)